<div style="background:linear-gradient(135deg,#0F3D6E 0%,#2176AE 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#C8DEF5;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 1 · CLASE 4</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: R², F-test, t-tests e Intervalos de Confianza</h1>
  <p style="color:#C8DEF5;font-size:16px;margin:0 0 24px 0;font-style:italic;">Del modelo estimado al informe con inferencia completa</p>
  <hr style="border-color:#5BA4CF;margin:0 0 20px 0;">
  <p style="color:#EAF2FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Tabla ANOVA desde cero

Dado el dataset generado abajo:
1. Estimar β̂ con lstsq
2. Calcular SST, SSR, SSE
3. Verificar SST = SSR + SSE
4. Construir la tabla ANOVA completa con GL y CM
5. ¿Qué porcentaje de varianza explica el modelo?

In [ ]:
np.random.seed(SEED)
n_e1, p_e1 = 120, 4
beta_e1 = np.array([8.0, 2.5, -1.0, 3.0])
X_e1 = np.column_stack([np.ones(n_e1),
         StandardScaler().fit_transform(np.random.randn(n_e1, p_e1-1))])
y_e1 = X_e1 @ beta_e1 + np.random.normal(0, 2.5, n_e1)

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_e1, p_e1 = 120, 4
beta_e1 = np.array([8.,2.5,-1.,3.])
X_e1 = np.column_stack([np.ones(n_e1),
         StandardScaler().fit_transform(np.random.randn(n_e1, p_e1-1))])
y_e1 = X_e1 @ beta_e1 + np.random.normal(0, 2.5, n_e1)

b_e1, *_ = np.linalg.lstsq(X_e1, y_e1, rcond=None)
yh_e1 = X_e1 @ b_e1; e_e1 = y_e1 - yh_e1
y_bar = y_e1.mean()
SST = ((y_e1-y_bar)**2).sum()
SSE = (e_e1**2).sum()
SSR = SST - SSE
df_r=p_e1-1; df_e=n_e1-p_e1; df_t=n_e1-1
MSR=SSR/df_r; MSE=SSE/df_e; F=MSR/MSE
pF=1-stats.f.cdf(F,df_r,df_e)

print(f'SST = {SST:.4f}'); print(f'SSR = {SSR:.4f}'); print(f'SSE = {SSE:.4f}')
print(f'SST == SSR+SSE: {np.isclose(SST,SSR+SSE)}')
print(f'\nR² = {1-SSE/SST:.4f}  ({(1-SSE/SST)*100:.1f}% varianza explicada)')
anova=pd.DataFrame({'Fuente':['Regresión','Residual','Total'],'SS':[SSR,SSE,SST],'GL':[df_r,df_e,df_t],'CM':[MSR,MSE,None],'F':[F,None,None],'p-val':[pF,None,None]})
print('\nTabla ANOVA:'); print(anova.to_string(index=False))

---
## 🟢 Ejercicio 2 — R² vs R² ajustado

Dado un dataset base, agregar features de ruido de 1 en 1 y mostrar:
1. Cómo evoluciona R² (siempre sube)
2. Cómo evoluciona R²_adj (baja con features inútiles)
3. ¿Cuántas features reales tiene el modelo?
4. ¿En qué punto R²_adj alcanza su máximo?

In [ ]:
np.random.seed(SEED)
n_e2 = 150
# Solo x1 y x2 tienen efecto real
x1 = np.random.randn(n_e2); x2 = np.random.randn(n_e2)
y_e2 = 3 + 2*x1 - 1.5*x2 + np.random.normal(0, 1.5, n_e2)
X_base = np.column_stack([np.ones(n_e2), x1, x2])

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_e2=150
x1=np.random.randn(n_e2); x2=np.random.randn(n_e2)
y_e2=3+2*x1-1.5*x2+np.random.normal(0,1.5,n_e2)
X_base=np.column_stack([np.ones(n_e2),x1,x2])
SST_e2=((y_e2-y_e2.mean())**2).sum()
r2_l,r2adj_l,p_l=[],[],[]
for extra in range(0,13):
    X_c=np.column_stack([X_base]+([np.random.randn(n_e2)] if extra>0 else [np.zeros((n_e2,0)).T.ravel()][:0]))  if extra==0 else \
        np.column_stack([X_base,np.random.randn(n_e2,extra)])
    if extra==0: X_c=X_base
    p_c=X_c.shape[1]
    b_c,*_=np.linalg.lstsq(X_c,y_e2,rcond=None)
    SSE_c=((y_e2-X_c@b_c)**2).sum()
    r2_l.append(1-SSE_c/SST_e2)
    r2adj_l.append(1-(SSE_c/(n_e2-p_c))/(SST_e2/(n_e2-1)))
    p_l.append(p_c)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(p_l,r2_l,'o-',color='#2176AE',lw=2,label='R²')
ax.plot(p_l,r2adj_l,'s-',color='#C0392B',lw=2,label='R² ajustado')
ax.axvline(3,color='gray',linestyle='--',lw=1.5,label='p=3 (modelo real)')
ax.set_xlabel('Número de parámetros p'); ax.set_ylabel('R²')
ax.set_title('R² vs R²_adj al agregar ruido', fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.25); plt.tight_layout(); plt.show()
best_p=p_l[np.argmax(r2adj_l)]
print(f'R²_adj máximo en p={best_p} (modelo real tiene p=3)  ✓')

---
## 🟡 Ejercicio 3 — t-tests manuales e interpretación

Con el dataset de ventas abajo:
1. Estimar β̂ y construir la tabla de coeficientes con t-stats y p-valores
2. Identificar qué features son significativas al 5%
3. Construir IC al 95% y al 99%
4. Interpretar en lenguaje de negocio cada coeficiente significativo
5. Si una feature no es significativa, ¿la eliminamos? ¿Por qué?

In [ ]:
np.random.seed(7)
n_e3 = 200
precio_e3    = np.random.uniform(5, 50, n_e3)
descuento_e3 = np.random.uniform(0, 0.4, n_e3)
competencia  = np.random.uniform(0, 10, n_e3)  # sin efecto real
trafico_e3   = np.random.gamma(4, 150, n_e3)

ventas_e3 = (500 - 8*precio_e3 + 300*descuento_e3
             + 0*competencia + 1.5*trafico_e3
             + np.random.normal(0, 120, n_e3))

feats_e3  = ['precio','descuento','competencia','trafico']
X_e3_raw  = np.column_stack([precio_e3, descuento_e3, competencia, trafico_e3])
X_e3 = np.column_stack([np.ones(n_e3), StandardScaler().fit_transform(X_e3_raw)])

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(7)
n_e3=200
p_e3=np.random.uniform(5,50,n_e3); d_e3=np.random.uniform(0,0.4,n_e3)
c_e3=np.random.uniform(0,10,n_e3); t_e3=np.random.gamma(4,150,n_e3)
ven_e3=500-8*p_e3+300*d_e3+0*c_e3+1.5*t_e3+np.random.normal(0,120,n_e3)
feats_e3=['precio','descuento','competencia','trafico']
X_e3=np.column_stack([np.ones(n_e3),StandardScaler().fit_transform(
    np.column_stack([p_e3,d_e3,c_e3,t_e3]))])
b_e3,*_=np.linalg.lstsq(X_e3,ven_e3,rcond=None)
yh_e3=X_e3@b_e3; e_e3=ven_e3-yh_e3
SST3=((ven_e3-ven_e3.mean())**2).sum(); SSE3=(e_e3**2).sum()
p3=X_e3.shape[1]; df_e3=n_e3-p3
s2_e3=SSE3/df_e3; XtX_inv3=np.linalg.inv(X_e3.T@X_e3)
SE_e3=np.sqrt(s2_e3*np.diag(XtX_inv3))
t_e3v=b_e3/SE_e3; pv_e3=2*(1-stats.t.cdf(np.abs(t_e3v),df=df_e3))
tc95=stats.t.ppf(0.975,df_e3); tc99=stats.t.ppf(0.995,df_e3)
nms=['intercepto']+feats_e3
print(f'R²={1-SSE3/SST3:.4f}\n')
print(f'{"Feature":14s} {"β̂":>8s} {"SE":>8s} {"t":>8s} {"p-val":>9s} {"IC95%":>20s} {"Sig":>5s}')
print('─'*78)
for i,nm in enumerate(nms):
    lo95,hi95=b_e3[i]-tc95*SE_e3[i],b_e3[i]+tc95*SE_e3[i]
    sig='***' if pv_e3[i]<0.001 else '**' if pv_e3[i]<0.01 else '*' if pv_e3[i]<0.05 else '(ns)'
    print(f'  {nm:12s} {b_e3[i]:>8.2f} {SE_e3[i]:>8.2f} {t_e3v[i]:>8.3f} {pv_e3[i]:>9.4f} [{lo95:>7.2f},{hi95:>7.2f}] {sig:>5s}')
print('\nFeature NO significativa: competencia (p>0.05)')
print('Decisión: mantenerla si hay razón teórica; eliminarla si es ruido puro.')

---
## 🔴 Ejercicio 4 — Reporte completo + selección de modelo

Implementá `seleccion_modelo(X, y, k=5)` que:
1. Prueba todas las combinaciones de k features
2. Para cada modelo calcula R²_adj y AIC = n·ln(SSE/n) + 2p
3. Retorna el Top 3 modelos por R²_adj
4. Para el mejor modelo, imprime el reporte completo con t-tests
5. Grafica R²_adj vs AIC para todos los modelos evaluados

In [ ]:
from itertools import combinations

np.random.seed(SEED)
n_e4 = 200
X_all_raw = np.random.randn(n_e4, 6)
beta_real_e4 = np.array([0, 2.0, 0, -1.5, 0, 1.0, 0])  # solo x1, x3, x5 reales
X_all = np.column_stack([np.ones(n_e4), StandardScaler().fit_transform(X_all_raw)])
y_e4  = X_all @ beta_real_e4 + np.random.normal(0, 1.5, n_e4)
feat_names_e4 = [f'x{i}' for i in range(1,7)]

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
from itertools import combinations
np.random.seed(SEED)
n_e4=200; X_all_raw=np.random.randn(n_e4,6)
beta_real_e4=np.array([0,2.,-1.5,0,1.,0,0])
X_all=np.column_stack([np.ones(n_e4),StandardScaler().fit_transform(X_all_raw)])
y_e4=X_all@beta_real_e4+np.random.normal(0,1.5,n_e4)
feat_names_e4=[f'x{i}' for i in range(1,7)]
SST_e4=((y_e4-y_e4.mean())**2).sum()
n4=n_e4

resultados=[]
for k in range(1,7):
    for combo in combinations(range(1,7), k):
        cols=list(combo); X_c=X_all[:,([0]+cols)]
        p_c=X_c.shape[1]
        b_c,*_=np.linalg.lstsq(X_c,y_e4,rcond=None)
        SSE_c=((y_e4-X_c@b_c)**2).sum()
        R2adj_c=1-(SSE_c/(n4-p_c))/(SST_e4/(n4-1))
        AIC_c=n4*np.log(SSE_c/n4)+2*p_c
        resultados.append({'features':tuple([feat_names_e4[c-1] for c in cols]),
                           'p':p_c,'R2adj':R2adj_c,'AIC':AIC_c})

df_res=pd.DataFrame(resultados).sort_values('R2adj',ascending=False)
print('TOP 5 modelos por R²_adj:')
print(df_res.head(5)[['features','p','R2adj','AIC']].to_string(index=False))

# Gráfico AIC vs R²adj
fig,ax=plt.subplots(figsize=(8,4.5))
sc=ax.scatter(df_res['AIC'],df_res['R2adj'],alpha=0.5,s=20,c=df_res['p'],cmap='viridis')
plt.colorbar(sc,label='# parámetros')
best=df_res.iloc[0]
ax.scatter(best['AIC'],best['R2adj'],color='#C0392B',s=150,zorder=5,label='Mejor modelo')
ax.set_xlabel('AIC (menor = mejor)'); ax.set_ylabel('R² ajustado (mayor = mejor)')
ax.set_title('Criterios de selección de modelo', fontweight='bold')
ax.legend(); ax.grid(True,alpha=0.2); plt.tight_layout(); plt.show()
print(f'\nMejor modelo: {best["features"]}  R²_adj={best["R2adj"]:.4f}  AIC={best["AIC"]:.2f}')

---
<div style="background:#EAF2FB;border-left:5px solid #27AE60;padding:16px 20px;border-radius:0 8px 8px 0;">

### ✅ Módulo 1 — Regresión Lineal completado

Cubriste el ciclo completo:
- **Clase 1:** Álgebra lineal, ecuación normal, hat matrix
- **Clase 2:** Eigenvalues, matrices aleatorias, distribución de β̂
- **Clase 3:** Supuestos Gauss-Markov, BLUE, diagnósticos
- **Clase 4:** R², ANOVA, F-test, t-tests, intervalos de confianza

</div>

---
<div style="background:#0F3D6E;color:white;padding:20px 24px;border-radius:8px;">
<strong>Siguiente módulo — Regresión Logística</strong><br>
Formulación desde el GLM · Función logit · Log-verosimilitud · Curvas ROC<br>
<em>Docente: Josef Rodriguez · Curso 8 · Modelos Estadísticos</em>
</div>